# SAÉ — Migration vers NoSQL

## Contexte et contraintes métier

La base décrit la programmation d’une radio et ses diffusions. Les requêtes s’appuient notamment sur les règles suivantes :

- **Protection du jeune public** : aucun titre `explicit` ne doit être diffusé pendant les tranches protégées du matin et de l’après-midi.
- **Quota de nouveautés** : l’antenne doit diffuser un minimum de titres récent·e·s, sélectionné·e·s selon la date de sortie de leur album.
- **Déclaration des diffusions** : pour les sociétés de gestion de droits, il faut pouvoir obtenir rapidement, pour chaque artiste, le nombre de diffusions et la durée cumulée correspondante.

Pour chaque requête, choisissez un ordre de présentation lorsqu’il n’est pas évident, afin de pouvoir comparer les résultats après migration.

In [2]:
!pip install pandas --quiet


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sqlite3
import pandas as pd
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

# Connection to Radio Kiosque database
sqlite_connection = sqlite3.connect("data/RadioKiosque.sqlite")

Comment requêter la base de données ?

In [2]:
pd.read_sql_query(
    """
        SELECT host, title
        FROM Shows
        ORDER BY host
    """,
    con=sqlite_connection
)

,host,title
0,Anouk Lindqvist,Le Réveil Kiosque
1,Anouk Lindqvist,Rush Hour
2,DJ Marvelle,Les Insomniaques
3,Dario Fenn,Tour de Piste
4,Françoise Dubreuil,Dimanche Vinyle
5,Karim Benali,Café Croissant
6,Karim Benali,Nouveautés Kiosque
7,Orla Quinn,Pause Midi
8,Orla Quinn,Weekend Matin
9,Salomé Ndiaye,Scène Ouverte


### 1. Auditeur·ices sans demande

Lister les auditeur·ices n’ayant jamais effectué de demande musicale.

In [5]:
# NOT EXISTS permet de conserver chaque auditeur une seule fois.
pd.read_sql_query("""
    SELECT l.listener_id, l.pseudo, l.country, l.city
    FROM Listeners AS l
    WHERE NOT EXISTS (
        SELECT 1
        FROM Requests AS r
        WHERE r.listener_id = l.listener_id
    )
    ORDER BY l.listener_id
    """
,con=sqlite_connection
)


,listener_id,pseudo,country,city
0,3,solar_raven63,Canada,Québec
1,7,red_seal79,France,Nantes
2,10,lunar_llama20,Belgique,Liège
3,12,pink_lynx87,Belgique,Bruxelles
4,14,red_owl90,Allemagne,Berlin
...,...,...,...,...
174,593,big_owl10,France,Lille
175,594,secret_sparrow56,Italie,Milan
176,596,fast_sparrow44,France,Grenoble
177,598,calm_raven24,Royaume-Uni,Londres


### 2. Bilan par animateur·ice

Pour chaque animateur·ice, indiquer le nombre d’émissions, le nombre de diffusions et la durée totale diffusée.

In [6]:
# COUNT(DISTINCT) évite de compter plusieurs fois une émission.
# La durée totale est exprimée en millisecondes.

pd.read_sql_query("""
    SELECT s.host AS animateur,
           COUNT(DISTINCT s.show_id) AS nb_emissions,
           COUNT(b.broadcast_id) AS nb_diffusions,
           COALESCE(SUM(t.duration_ms), 0) AS duree_totale_ms
    FROM Shows AS s
    LEFT JOIN Broadcasts AS b ON b.show_id = s.show_id
    LEFT JOIN Tracks AS t ON t.track_id = b.track_id
    GROUP BY s.host
    ORDER BY s.host
    """,
    con=sqlite_connection
)

,animateur,nb_emissions,nb_diffusions,duree_totale_ms
0,Anouk Lindqvist,2,43894,10140191003
1,DJ Marvelle,1,33201,7538676031
2,Dario Fenn,1,21830,5080002640
3,Françoise Dubreuil,1,6466,1510922539
4,Karim Benali,2,28522,6629138076
5,Orla Quinn,2,34592,8107701472
6,Salomé Ndiaye,2,17788,4045055699
7,Yasmine Tazi,1,13814,3016880054


### 3. Bilan par tranche horaire

Pour chaque tranche horaire, indiquer le nombre de diffusions et le nombre de titres explicites diffusés.

In [7]:
# CASE WHEN compte les diffusions de titres explicites.
# Un titre diffusé trois fois est compté trois fois.

pd.read_sql_query("""
    SELECT s.time_slot AS tranche_horaire,
           COUNT(b.broadcast_id) AS nb_diffusions,
           SUM(CASE WHEN t.explicit = 1
                    THEN 1 ELSE 0 END) AS nb_titres_explicites
    FROM Shows AS s
    LEFT JOIN Broadcasts AS b ON b.show_id = s.show_id
    LEFT JOIN Tracks AS t ON t.track_id = b.track_id
    GROUP BY s.time_slot
    ORDER BY s.time_slot
    """,
    con=sqlite_connection
)

,tranche_horaire,nb_diffusions,nb_titres_explicites
0,afternoon,49981,502
1,evening,51766,20210
2,morning,56270,566
3,night,42090,11729


### 4. Bilan par titre

Pour chaque titre, indiquer le nombre de diffusions, l’audience cumulée et le nombre d’émissions différentes où il a été programmé.

In [8]:
# LEFT JOIN conserve les titres jamais diffusés.
# COUNT(DISTINCT) compte les émissions différentes.

pd.read_sql_query("""
    SELECT t.track_id,
           t.name AS titre,
           COUNT(b.broadcast_id) AS nb_diffusions,
           COALESCE(SUM(b.listeners), 0) AS audience_cumulee,
           COUNT(DISTINCT b.show_id) AS nb_emissions
    FROM Tracks AS t
    LEFT JOIN Broadcasts AS b ON b.track_id = t.track_id
    GROUP BY t.track_id, t.name
    ORDER BY t.track_id
    """,
    con=sqlite_connection
)

,track_id,titre,nb_diffusions,audience_cumulee,nb_emissions
0,000d0lQMYaRR5ZXS9nTeiN,Paris sous les bombes,0,0,0
1,001op3H5t7sehq5JCG2PvM,One More Time,0,0,0
2,004XgceMTFqLTYVt5FQEcO,Notre Dame de Paris,16,2125,7
3,0074XGa4ytWx2s1o48jO3y,Boîte à Gants,0,0,0
4,00A0Cq31pUC4HlG5WpKjIW,Toast 🍸 - Extended Version,28,4137,11
...,...,...,...,...,...
11330,7zscdQe9CjzXnqT3P1Ey7K,If You Want Blood (You've Got It),0,0,0
11331,7zuoFc4wKjcd0Y4eLPUEvh,We're Looking for the Whales - Live,53,7549,10
11332,7zvwxSE35TwHoXiolyjRDX,Mine Right Now,0,0,0
11333,7zxFVo238IlUXMuIDh6lLz,L’Épine,0,0,0


### 5. Bilan par artiste

Pour chaque artiste (principal·e ou invité·e), indiquer le nombre de titres au catalogue, le nombre de diffusions et la durée cumulée diffusée. Inclure les artistes jamais diffusé·es. Trier par ordre décroissant du nombre de diffusions.

In [9]:
# TrackArtists relie les artistes à leurs titres.
# On inclut les artistes invités et ceux jamais diffusés.
# Chaque passage ajoute la durée du titre à l'artiste concerné.

pd.read_sql_query("""
    SELECT a.artist_id,
           a.name AS artiste,
           COUNT(DISTINCT ta.track_id) AS nb_titres,
           COUNT(b.broadcast_id) AS nb_diffusions,
           COALESCE(SUM(
               CASE WHEN b.broadcast_id IS NOT NULL
                    THEN t.duration_ms ELSE 0 END
           ), 0) AS duree_cumulee_ms
    FROM Artists AS a
    LEFT JOIN TrackArtists AS ta ON ta.artist_id = a.artist_id
    LEFT JOIN Tracks AS t ON t.track_id = ta.track_id
    LEFT JOIN Broadcasts AS b ON b.track_id = ta.track_id
    GROUP BY a.artist_id, a.name
    ORDER BY nb_diffusions DESC, a.artist_id
    """,
    con=sqlite_connection
)

,artist_id,artiste,nb_titres,nb_diffusions,duree_cumulee_ms
0,2ye2Wgw4gimLv2eAKyk1NB,Metallica,246,4495,1293246523
1,1WPcVNert9hn7mHsPKDn7j,Édith Piaf,167,3377,664359973
2,4ko6Ysxtvx9EY9GEFslrIz,Suprême NTM,157,3197,670790626
3,2jzc5TC5TVFLXQlBNiIUzE,a-ha,87,2771,736001470
4,01C9OoXDvCKkGcf735Tcfo,Serge Gainsbourg,99,2662,474314887
...,...,...,...,...,...
1587,7x3f7c0fBanNlQwpx1255g,Françoise Hardy,1,0,0
1588,7x5rK9BClDQ8wmCkYAGsQp,Paolo Nutini,1,0,0
1589,7xQTON6uj7akAGzlanUzy3,Charlie Winston,1,0,0
1590,7xZX46EPiYL9qfO6cb0sVI,Tom Liljegren,1,0,0


### 6. Diffusions par décennie et tranche horaire

Présenter le nombre de diffusions sous forme de tableau croisé entre la décennie de sortie de l’album et la tranche horaire.

Voici le format attendu :

| decade | morning | afternoon | evening | night |
|-------:|--------:|----------:|--------:|------:|
|   1960 |         |           |         |       |
|   1970 |         |           |         |       |
|   1980 |         |           |         |       |
|   1990 |         |           |         |       |
|   2000 |         |           |         |       |
|   2010 |         |           |         |       |
|   2020 |         |           |         |       |

&#128161; Exécuter la requête en SQL avec la décennie, la tranche horaire et le nombre de diffusion. Puis pivoter la table avec la fonction `pivot_table` de Pandas.

In [11]:
# strftime extrait l'année de sortie de l'album.
# La division par 10 permet d'obtenir la décennie.
# CASE WHEN construit directement le tableau croisé. Mieux que Pivot Table?

pd.read_sql_query("""
    SELECT
        CAST(
            CAST(strftime('%Y', a.release_date) AS INTEGER) / 10
            AS INTEGER
        ) * 10 AS decade,

        SUM(CASE WHEN s.time_slot = 'morning'
                 THEN 1 ELSE 0 END) AS morning,

        SUM(CASE WHEN s.time_slot = 'afternoon'
                 THEN 1 ELSE 0 END) AS afternoon,

        SUM(CASE WHEN s.time_slot = 'evening'
                 THEN 1 ELSE 0 END) AS evening,

        SUM(CASE WHEN s.time_slot = 'night'
                 THEN 1 ELSE 0 END) AS night

    FROM Broadcasts AS b
    JOIN Tracks AS t ON t.track_id = b.track_id
    JOIN Albums AS a ON a.album_id = t.album_id
    JOIN Shows AS s ON s.show_id = b.show_id

    GROUP BY decade
    ORDER BY decade
    """,
    con=sqlite_connection
)

,decade,morning,afternoon,evening,night
0,1960,1390,1236,813,779
1,1970,1833,1652,1107,1004
2,1980,3343,2958,1859,1833
3,1990,5636,5084,5164,4300
4,2000,10587,9374,6354,6026
5,2010,17229,15392,16936,13819
6,2020,16252,14285,19533,14329


### 7. Audience par décennie et tranche horaire

Présenter l’audience cumulée sous forme de tableau croisé entre la décennie de sortie de l’album et la tranche horaire.

In [12]:
# Même principe que Q6, mais on additionne l'audience
# de chaque diffusion plutôt que de compter les passages.

pd.read_sql_query("""
    SELECT
        CAST(
            CAST(strftime('%Y', a.release_date) AS INTEGER) / 10
            AS INTEGER
        ) * 10 AS decade,

        SUM(CASE WHEN s.time_slot = 'morning'
                 THEN b.listeners ELSE 0 END) AS morning,

        SUM(CASE WHEN s.time_slot = 'afternoon'
                 THEN b.listeners ELSE 0 END) AS afternoon,

        SUM(CASE WHEN s.time_slot = 'evening'
                 THEN b.listeners ELSE 0 END) AS evening,

        SUM(CASE WHEN s.time_slot = 'night'
                 THEN b.listeners ELSE 0 END) AS night

    FROM Broadcasts AS b
    JOIN Tracks AS t ON t.track_id = b.track_id
    JOIN Albums AS a ON a.album_id = t.album_id
    JOIN Shows AS s ON s.show_id = b.show_id

    GROUP BY decade
    ORDER BY decade
    """,
    con=sqlite_connection
)

,decade,morning,afternoon,evening,night
0,1960,322812,205118,108856,28278
1,1970,401761,265248,142185,35318
2,1980,754338,484825,244668,65035
3,1990,1252052,826204,672995,151936
4,2000,2354505,1527890,815377,209362
5,2010,3853223,2500796,2190868,488681
6,2020,3628210,2332811,2507266,503677


### 8. Top 10 des audiences moyennes

Parmi les titres diffusés au moins trois fois, lister les dix titres ayant la plus forte audience moyenne par diffusion. Trier par l'audience moyenne.

In [13]:
# AVG calcule l'audience moyenne par diffusion.
# HAVING conserve uniquement les titres diffusés au moins 3 fois.

pd.read_sql_query("""
    SELECT t.track_id,
           t.name AS titre,
           COUNT(b.broadcast_id) AS nb_diffusions,
           AVG(b.listeners) AS audience_moyenne
    FROM Tracks AS t
    JOIN Broadcasts AS b ON b.track_id = t.track_id
    GROUP BY t.track_id, t.name
    HAVING COUNT(b.broadcast_id) >= 3
    ORDER BY audience_moyenne DESC, t.track_id
    LIMIT 10
    """,
    con=sqlite_connection
)

,track_id,titre,nb_diffusions,audience_moyenne
0,7nRCeBY3Oyqu94Nav3ViTT,Miroirs,17,210.529412
1,4uIXvy8PbYLEhs0PZgaP8p,Delirium (feat. Mark E. Smith),13,209.769231
2,65fMfPqIzVV9XAzQ4UvkmN,Elle ecoute pousser les fleurs - Remastered,9,209.111111
3,7nzsY8vlnKdvGOEE0rjAXZ,Arabella,15,206.933333
4,2LKRpB6YTz0MUqbhJoqNCg,Back It Up,15,206.733333
5,2Hb8HUCnMZ74gwl4DppYDd,Sampha's Plea,797,206.424090
6,76HKOVWqsEg26SECtmw7Rz,The Way You Make Me Feel - 2012 Remaster,12,205.666667
7,40o1yxXRNZqbKLQOqFJKp0,RA TA TA,25,204.840000
8,6H3kDe7CGoWYBabAeVWGiD,Gimme Shelter,666,202.632132
9,720ALyzJaCOQRVxNMdjwCO,Dors,17,201.705882


### 9. Infractions à la protection du jeune public

Lister chaque diffusion d’un titre explicite dans une émission protégée, avec le titre, l’identifiant de l’émission et l’horodatage.

In [14]:
# explicit = 1 désigne un titre explicite.
# is_protected = 1 désigne une émission protégée.

pd.read_sql_query("""
    SELECT b.broadcast_id,
           t.name AS titre,
           s.show_id AS id_emission,
           b.played_at AS horodatage
    FROM Broadcasts AS b
    JOIN Tracks AS t ON t.track_id = b.track_id
    JOIN Shows AS s ON s.show_id = b.show_id
    WHERE t.explicit = 1
      AND s.is_protected = 1
    ORDER BY b.played_at, b.broadcast_id
    """,
    con=sqlite_connection
)

,broadcast_id,titre,id_emission,horodatage
0,185,GOD.,11,2023-10-08 11:46:02
1,404,Je réponds plus,1,2023-10-10 08:30:50
2,590,3 PALI,1,2023-10-11 07:14:02
3,925,Criminel,4,2023-10-12 16:49:48
4,1115,Red Bottom Sky,4,2023-10-13 15:10:52
...,...,...,...,...
1063,199219,DIEU MERCI RELOADED,2,2026-09-29 09:06:28
1064,199261,Ma cousin,3,2026-09-29 12:57:55
1065,199283,Yusuf,4,2026-09-29 15:26:47
1066,199508,Trop beau - live acoustique,4,2026-09-30 16:12:11


### 10. Albums diffusés en entier

Lister les albums dont tous les titres ont été diffusés au moins une fois.

In [15]:
# On récupère d'abord les titres diffusés au moins une fois.
# HAVING vérifie que tous les titres de chaque album
# sont présents dans cette liste.

pd.read_sql_query("""
    WITH titres_diffuses AS (
        SELECT DISTINCT track_id
        FROM Broadcasts
    )

    SELECT a.album_id,
           a.name AS album
    FROM Albums AS a
    JOIN Tracks AS t ON t.album_id = a.album_id
    LEFT JOIN titres_diffuses AS d ON d.track_id = t.track_id

    GROUP BY a.album_id, a.name
    HAVING COUNT(t.track_id) = COUNT(d.track_id)
    ORDER BY a.album_id
    """,
    con=sqlite_connection
)

,album_id,album
0,014U2yqVgeCNqykPGzgyki,Midnight Sun
1,0488X5veBK6t3vSmIiTDJY,Before You Go
2,05OLq8nG4L5WYZXTfXVYM7,Nightcall
3,06ZMpecsvvoHGWJHlc2St7,Brividi
4,07S8hnczxCG1wvNMjbq8An,VIVE LA MONNAIE
...,...,...
215,7ttqdPpQs5GqpjirAusheg,Рута
216,7u7l9XNKOdM1UXLOL4BJq8,Night Trouble
217,7vDWBZVTaeyqbveurnGBvs,Mãe
218,7zXG62SwOcTipKVKcdonsM,Alone


### 11. Podcasts classés par pays

Pour chaque pays, indiquer le nombre de podcasts classés, le nombre de podcasts explicites et le nombre de podcasts dont la fiche est fiable.

In [16]:
# COUNT(DISTINCT) évite de compter plusieurs fois un podcast.
# Une fiche fiable possède le statut 'exact' dans cette base.

pd.read_sql_query("""
    SELECT r.country AS pays,
           COUNT(DISTINCT r.podcast_id) AS nb_podcasts_classes,

           COUNT(DISTINCT CASE WHEN p.explicit = 1
                 THEN r.podcast_id END) AS nb_podcasts_explicites,

           COUNT(DISTINCT CASE WHEN p.match_status = 'exact'
                 THEN r.podcast_id END) AS nb_fiches_fiables

    FROM PodcastRankings AS r
    JOIN Podcasts AS p ON p.podcast_id = r.podcast_id
    GROUP BY r.country
    ORDER BY r.country
    """,
    con=sqlite_connection
)

,pays,nb_podcasts_classes,nb_podcasts_explicites,nb_fiches_fiables
0,AT,100,34,100
1,BE,100,30,100
2,CH,100,33,100
3,DE,100,37,100
4,DK,100,47,100
5,ES,100,29,100
6,FI,100,55,100
7,FR,101,29,101
8,GB,100,70,100
9,IE,100,63,100


### 12. Podcasts programmables

Lister les podcasts programmables, avec leur nom, leur éditeur, le nombre de pays dans le top 10 desquels ils figurent et leur meilleur rang. Trier du plus présent au moins présent, puis par meilleur rang.

In [17]:
# Un podcast programmable doit être non explicite,
# avoir une fiche fiable et figurer dans le top 10 d'au moins un pays.
# MIN(rank) donne son meilleur classement.

pd.read_sql_query("""
    SELECT p.podcast_id,
           p.name AS podcast,
           p.publisher AS editeur,
           COUNT(DISTINCT r.country) AS nb_pays_top10,
           MIN(r.rank) AS meilleur_rang

    FROM Podcasts AS p
    JOIN PodcastRankings AS r ON r.podcast_id = p.podcast_id

    WHERE p.explicit = 0
      AND p.match_status = 'exact'
      AND r.rank BETWEEN 1 AND 10

    GROUP BY p.podcast_id, p.name, p.publisher
    ORDER BY nb_pays_top10 DESC, meilleur_rang ASC, p.podcast_id
    """,
    con=sqlite_connection
)

,podcast_id,podcast,editeur,nb_pays_top10,meilleur_rang
0,0xVTjXuo1jKKi2lyv0cRmI,Aha! Zehn Minuten Alltags-Wissen,WELT,4,1
1,4xQ0IUSsrfs6pltTnwr3Kk,LEGEND,Guillaume Pley,4,5
2,2HGcJRYrjGnpce6bRp8UXm,The Psychology of your 20s,iHeartPodcasts,3,1
3,4us7Lec3iNeAXDoG56Vths,5 Minuten Harry Podcast von Coldmirror,funk - von ARD und ZDF,3,5
4,2OyqGF8rokIi7K0CDwQc1Z,Ray William Johnson: True Story Podcast,Ray William Johnson,3,8
5,1Tn0rNAr68DU0gLZXWiptV,CHRONIQUES CRIMINELLES,Jacques Pradel - TF1,2,2
6,1HCF9OMp8ItCvFT1P3rC7x,Terra X History - Der Podcast,ZDF - Terra X,2,4
7,7iQXmUT7XGuZSzAMjoNWlX,The Diary Of A CEO with Steven Bartlett,FlightStory,2,6
8,16zDAJSEJp06lZnPdHv8pI,Die Deutschen Podcast,Nizar und Shayan,2,7
9,7AH8nT1CoTmV1MfhxyTl3n,SAFE PACE - Le podcast des sports d'endurance,"Hugo Clément, Charlotte Morel, Hugo Tormento e...",2,7


### 13. Durée et activité des podcasts programmables

Pour chaque podcast programmable, indiquer le nombre d’épisodes connus, la durée moyenne de ses épisodes (en minutes) et la date de son dernier épisode. Ne garder que les podcasts dont la durée moyenne ne dépasse pas une heure.

In [18]:
# EXISTS vérifie que le podcast figure dans un top 10.
# AVG calcule la durée moyenne des épisodes.
# La division par 60000 convertit les millisecondes en minutes.
# HAVING limite la durée moyenne à 60 minutes.

pd.read_sql_query("""
    SELECT p.podcast_id,
           p.name AS podcast,
           COUNT(e.episode_id) AS nb_episodes_connus,
           AVG(e.duration_ms) / 60000.0 AS duree_moyenne_minutes,
           MAX(e.release_date) AS date_dernier_episode

    FROM Podcasts AS p
    LEFT JOIN PodcastEpisodes AS e ON e.podcast_id = p.podcast_id

    WHERE p.explicit = 0
      AND p.match_status = 'exact'
      AND EXISTS (
          SELECT 1
          FROM PodcastRankings AS r
          WHERE r.podcast_id = p.podcast_id
            AND r.rank BETWEEN 1 AND 10
      )

    GROUP BY p.podcast_id, p.name
    HAVING AVG(e.duration_ms) <= 3600000
    ORDER BY p.name, p.podcast_id
    """,
    con=sqlite_connection
)

,podcast_id,podcast,nb_episodes_connus,duree_moyenne_minutes,date_dernier_episode
0,6wSE4lw2azBvT1Zfoq0yIB,15 Minuten. Der tagesschau-Podcast am Morgen,5,15.256617,2026-10-16
1,4us7Lec3iNeAXDoG56Vths,5 Minuten Harry Podcast von Coldmirror,5,28.216990,2018-10-10
2,0kRvNOwfw2iAxZiPo4fbi9,AD Voetbal podcast,5,39.859020,2026-10-18
3,0LQrN9BnTKfMPrj9z4aH8F,AVHØRT,5,52.324913,2026-10-14
4,0xVTjXuo1jKKi2lyv0cRmI,Aha! Zehn Minuten Alltags-Wissen,5,22.167503,2026-10-16
5,7rgFvrfap2bLcItGw45QZH,Animales Humanos Podcast,5,41.554270,2026-10-17
6,6X3TkpZ2aXY6ggc7fJRPZK,BÅLSNAK,5,53.863437,2026-10-11
7,1Tn0rNAr68DU0gLZXWiptV,CHRONIQUES CRIMINELLES,5,55.896453,2026-10-17
8,5Yl4ao85LeyLc56nvm1E2T,Café Com Deus Pai | Podcast oficial,5,4.645120,2026-10-18
9,55S7Dxlkin3BCSXrKUvbha,Crimes e Mistérios Reais,5,44.424613,2025-10-06


In [19]:
# Close the connection to the database
sqlite_connection.close()